## Load the Cleaned Dataset

In [11]:
import pandas as pd
df = pd.read_csv(
    "../data/processed/cleaned_demand_timeseries.csv",
    parse_dates = [0],
    index_col = 0
)

df.head()

,demand,demand_diff
timestamp,,
2023-01-01 00:00:00,18995,NaN
2023-01-01 00:30:00,19730,735.0
2023-01-01 01:00:00,19327,-403.0
2023-01-01 01:30:00,18589,-738.0
2023-01-01 02:00:00,17806,-783.0


## Creat Lag features

In [12]:
# Previous half hour
df["lag_1"] = df["demand"].shift(1)

# Same time previous day
df["lag_48"] = df["demand"].shift(48)

df[["demand", "lag_1", "lag_48"]].head(60)

,demand,lag_1,lag_48
timestamp,,,
2023-01-01 00:00:00,18995,NaN,NaN
2023-01-01 00:30:00,19730,18995.0,NaN
2023-01-01 01:00:00,19327,19730.0,NaN
2023-01-01 01:30:00,18589,19327.0,NaN
2023-01-01 02:00:00,17806,18589.0,NaN
2023-01-01 02:30:00,17105,17806.0,NaN
2023-01-01 03:00:00,16459,17105.0,NaN
2023-01-01 03:30:00,15691,16459.0,NaN
2023-01-01 04:00:00,15221,15691.0,NaN


## Create Rolling Statistics

In [13]:
# Rolling mean (daily window)
df["rolling_mean_48"] = df["demand"].rolling(window=48).mean()

df[["demand", "rolling_mean_48"]].head(100)


,demand,rolling_mean_48
timestamp,,
2023-01-01 00:00:00,18995,NaN
2023-01-01 00:30:00,19730,NaN
2023-01-01 01:00:00,19327,NaN
2023-01-01 01:30:00,18589,NaN
2023-01-01 02:00:00,17806,NaN
...,...,...
2023-01-02 23:30:00,21625,28047.125000
2023-01-03 00:00:00,22500,28066.916667
2023-01-03 00:30:00,23043,28083.687500


## Extract Date & Time Components

In [14]:
# Critical features for XGBoost and LSTM
dt_idx = pd.DatetimeIndex(df.index)

df["hour"] = dt_idx.hour
df["day_of_week"] = dt_idx.dayofweek
df["month"] = dt_idx.month

df[["hour", "day_of_week", "month"]].head()


,hour,day_of_week,month
timestamp,,,
2023-01-01 00:00:00,0,6,1
2023-01-01 00:30:00,0,6,1
2023-01-01 01:00:00,1,6,1
2023-01-01 01:30:00,1,6,1
2023-01-01 02:00:00,2,6,1


## Drop rows with NaNs (Created by Lags)

In [15]:
df_fe = df.dropna()

df_fe.isna().sum()

demand             0
demand_diff        0
lag_1              0
lag_48             0
rolling_mean_48    0
hour               0
day_of_week        0
month              0
dtype: int64

## Save Feature-Engineered Dataset

In [16]:
# Save the fULL dataset (Unscaled, No Split)
# This file will be the input for BOTH XGBoost and LSTM later.
df_fe.to_csv("../data/processed/feature_engineered_unscaled.csv")

print("Feature engineering complete. File saved to: ../data/processed/feature_engineered_unscaled.csv")
print(f"Shape: {df_fe.shape}")

Feature engineering complete. File saved to: ../data/processed/feature_engineered_unscaled.csv
Shape: (282096, 8)
